# 신규 보고서 수집 → JSON 변환 → 임베딩 (증분 · 병렬 최적화)

KOSPI·KOSDAQ 전종목 중 **미보유** **2025 사업보고서(2025.12)** + **2026 1분기보고서(2026.03)** 를
DART 수집 → 청크 JSON → BGE-M3 임베딩 → ChromaDB upsert.

**최적화(v2)**: 병목인 DART 네트워크(목록·문서 다운로드)를 스레드 병렬(N_WORKERS), 임베딩은
웨이브 단위로 청크를 모아 큰 배치로 처리(단, upsert는 5000 단위 분할). resumable.

**주의**: 먼저 TEST_LIMIT(기본 5)로 검증 후 None 으로 전체 실행. 전체는 수 시간 + 수천 API 콜.
전체 실행 전 챗봇·valuation 서버를 내려 GPU(8GB) 경합을 피하세요.


In [ ]:
# ── 1) 셋업 ───────────────────────────────────────────────────────────
import os, sys, time, json, re, io, zipfile, random, threading
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import xml.etree.ElementTree as ET
import requests

VAR_ROOT = Path(r"C:\Users\Admin\Desktop\VAR")     # 환경 확인
sys.path.insert(0, str(VAR_ROOT))
from dotenv import load_dotenv
load_dotenv(VAR_ROOT / ".env")
DART_KEY = os.getenv("DART_API_KEY"); assert DART_KEY, "DART_API_KEY 미설정"

from valuation_engine.report_ingest import download_document, parse_chunks
from valuation_engine.report_detector import list_periodic_reports
from embedding.embedder import embed_texts
from embedding.vector_store import add_batch, get_existing_ids, get_stats

KOSPI_DIR, KOSDAQ_DIR = VAR_ROOT / "KOSPI", VAR_ROOT / "KOSDAQ"

# 파라미터(튜닝) — DART 재차단 방지를 위해 동시성 낮춤 + 전역 스로틀
TARGET_PERIODS = {"2025.12", "2026.03"}      # 2025 사업 + 2026 1분기
LIST_BGN, LIST_END = "20260101", "20260527"
N_WORKERS     = 8        # DART 동시 호출. 전역 스로틀이 총량을 캡하므로 안전(처리량 확보용)
WAVE_SIZE     = 40       # 임베딩 1회에 모을 보고서 수
UPSERT_BATCH  = 4000     # ChromaDB upsert/임베딩 분할 크기(최대 5461 미만 필수)
TEST_LIMIT    = None     # 검증용 앞 N종목. 전체는 None
REQUEST_DELAY = 0.1      # DART 호출 간 최소 간격(초) — 전역 스로틀. 분당 600회(한도 1000회의 60%, 마진 포함)
PROGRESS_LOG  = VAR_ROOT / "embedding" / "collect_progress.json"

_PERIOD = {"12": ("annual","annual","사업보고서"), "03": ("q1","quarterly","분기보고서"),
           "06": ("semiannual","semiannual","반기보고서"), "09": ("q3","quarterly","분기보고서")}

# 전역 스로틀: 모든 워커가 공유 — DART 호출이 REQUEST_DELAY 간격 이상 벌어지도록 직렬화
_throttle_lock = threading.Lock()
_last_call = [0.0]
def _throttle():
    with _throttle_lock:
        dt = time.time() - _last_call[0]
        if dt < REQUEST_DELAY:
            time.sleep(REQUEST_DELAY - dt)
        _last_call[0] = time.time()

def with_retry(fn, *a, tries=5, base=1.0, **k):
    # DART 호출 재시도(전역 스로틀 + 지수 백오프 + 지터). 연결 끊김(RemoteDisconnected)에도 견고.
    for t in range(tries):
        try:
            _throttle()
            return fn(*a, **k)
        except Exception:
            if t == tries - 1:
                raise
            time.sleep(base * (2 ** t) + random.uniform(0, 0.5))

print("DART OK |", get_stats())


In [ ]:
# ── 2) 종목 유니버스 (CSV 2개, cp949) ────────────────────────────────
import csv
def load_universe():
    files = {"KOSPI":  r"C:\Users\Admin\Downloads\data_1426_20260527.csv",
             "KOSDAQ": r"C:\Users\Admin\Downloads\data_1439_20260527.csv"}
    rows, seen = [], set()
    for market, p in files.items():
        with open(p, encoding="cp949", newline="") as f:
            for r in csv.DictReader(f):
                code = (r.get("단축코드") or "").strip().zfill(6)
                if (code and (r.get("증권구분") or "").strip() == "주권"
                        and (r.get("주식종류") or "").strip() == "보통주" and code not in seen):
                    seen.add(code)
                    rows.append({"stock_code": code, "market": market,
                                 "corp_name": (r.get("한글 종목약명") or "").strip()})
    return rows
universe = load_universe()
print("수집 대상 종목:", len(universe))


In [ ]:
# ── 3) corp_code 매핑 (corpCode.xml — 세션+재시도+로컬캐시) ───────────
from requests.adapters import HTTPAdapter
try:
    from urllib3.util.retry import Retry
except Exception:
    Retry = None
CORPCODE_CACHE = VAR_ROOT / "embedding" / "corpcode.xml"   # 1회 받으면 재사용(재실행시 재다운로드X)

def build_corpcode_map():
    if CORPCODE_CACHE.exists() and CORPCODE_CACHE.stat().st_size > 100000:
        xmlb = CORPCODE_CACHE.read_bytes()
    else:
        s = requests.Session()
        if Retry:
            s.mount("https://", HTTPAdapter(max_retries=Retry(
                total=6, backoff_factor=1.5,
                status_forcelist=[429,500,502,503,504], allowed_methods=["GET"])))
        last = None
        for attempt in range(6):
            try:
                r = s.get("https://opendart.fss.or.kr/api/corpCode.xml",
                          params={"crtfc_key": DART_KEY}, timeout=300)
                r.raise_for_status()
                z = zipfile.ZipFile(io.BytesIO(r.content))
                xmlb = z.read(z.namelist()[0])
                CORPCODE_CACHE.write_bytes(xmlb)   # 캐시 저장
                break
            except Exception as e:
                last = e
                print(f"  corpCode 재시도 {attempt+1}/6 — {str(e)[:80]}")
                time.sleep(2 * (attempt + 1))
        else:
            raise RuntimeError(f"corpCode.xml 다운로드 실패(재시도 소진): {last}")
    root = ET.fromstring(xmlb)
    m = {}
    for el in root.iter("list"):
        sc=(el.findtext("stock_code") or "").strip(); cc=(el.findtext("corp_code") or "").strip()
        cn=(el.findtext("corp_name") or "").strip()
        if sc and cc: m[sc.zfill(6)] = {"corp_code": cc.zfill(8), "corp_name": cn}
    return m

CORP = build_corpcode_map()
print("corp_code 매핑:", len(CORP), "| 캐시:", CORPCODE_CACHE)


In [ ]:
# ── 4) 헬퍼: 보유여부 · 대상탐색 · 다운로드+파싱(스레드용) ───────────
def jsonl_path(market, stock, corp_name, year, plabel):
    safe = re.sub(r'[\\/:*?"<>|]+', "", corp_name).strip() or stock
    d = KOSPI_DIR if market == "KOSPI" else KOSDAQ_DIR
    return d / f"{stock}_{safe}_{year}_{plabel}_chunks.jsonl"

def already_have(market, stock, year, plabel):
    d = KOSPI_DIR if market == "KOSPI" else KOSDAQ_DIR
    return any(d.glob(f"{stock}_*_{year}_{plabel}_chunks*.jsonl"))

def fetch_targets(rec):
    # (스레드) 정기공시 목록 → 대상 보고서. 반환 (rec, [rep...])
    cc = CORP[rec["stock_code"]]["corp_code"]
    reps = with_retry(list_periodic_reports, cc, LIST_BGN, LIST_END)
    return rec, [r for r in reps if r["period"] in TARGET_PERIODS]

def collect_report(rec, rep):
    # (스레드) document.xml 다운로드 → 파싱 → 메타보강 → jsonl 저장. 임베딩은 메인에서 일괄.
    stock, market = rec["stock_code"], rec["market"]
    cc, cn = CORP[stock]["corp_code"], CORP[stock]["corp_name"]
    plabel, rtype, prefix = _PERIOD[rep["month"]]
    year = rep["year"]; rkind = f"{year}-{plabel}"; rnm = f"{prefix} ({rep['period']})"
    src = f"https://dart.fss.or.kr/dsaf001/main.do?rcpNo={rep['rcept_no']}"
    meta = {"ticker": stock, "corp_name": cn, "corp_code": cc, "year": year,
            "rcept_no": rep["rcept_no"], "rcept_dt": rep["rcept_dt"], "period_label": plabel}
    xml = with_retry(download_document, rep["rcept_no"])
    chunks = parse_chunks(xml, meta)
    recs = []
    for c in chunks:
        md_ = c["metadata"]
        md_.update({"report_type": rtype, "report_kind": rkind, "report_nm": rnm,
                    "group": market, "source_url": src})
        recs.append({"id": c["id"], "group": market, "stock_code": stock, "corp_code": cc,
                     "corp_name": cn, "report_nm": rnm, "report_kind": rkind, "report_type": rtype,
                     "rcept_no": rep["rcept_no"], "rcept_dt": rep["rcept_dt"], "fiscal_period": rkind,
                     "source_url": src, "parse_mode": md_.get("parse_mode","ingest_v1"),
                     "kind": md_["kind"], "section_path": [s for s in (md_["section_main"], md_["section_sub"]) if s],
                     "section_path_str": md_["section_path_str"], "char_len": md_["char_len"], "text": c["text"]})
    return {"key": f"{stock}:{year}-{plabel}", "market": market, "stock": stock,
            "corp_name": cn, "year": year, "plabel": plabel, "records": recs, "chunks": chunks}


In [ ]:
# ── 5) 메인: 병렬 discovery → 웨이브(병렬 다운로드 + 분할 임베딩) ─────
progress = json.loads(PROGRESS_LOG.read_text(encoding="utf-8")) if PROGRESS_LOG.exists() else {}
fails = []
t0 = time.time()
work_src = universe if TEST_LIMIT is None else universe[:TEST_LIMIT]
work_src = [r for r in work_src if r["stock_code"] in CORP]

# Phase 1: 정기공시 목록 병렬 조회 → 작업항목
print(f"[1/2] 대상 탐색 (병렬 {N_WORKERS}) — 종목 {len(work_src)}…")
work = []
with ThreadPoolExecutor(N_WORKERS) as ex:
    futs = [ex.submit(fetch_targets, rec) for rec in work_src]
    for j, fut in enumerate(as_completed(futs), 1):
        try:
            rec, reps = fut.result()
        except Exception as e:
            fails.append(("list", str(e)[:120])); continue
        for rep in reps:
            plabel = _PERIOD[rep["month"]][0]
            key = f"{rec['stock_code']}:{rep['year']}-{plabel}"
            if progress.get(key) == "done":
                continue
            if already_have(rec["market"], rec["stock_code"], rep["year"], plabel):
                progress[key] = "done"; continue
            work.append((rec, rep))
        if j % 200 == 0:
            print(f"   탐색 {j}/{len(futs)}…")
print(f"   → 수집 대상 보고서 {len(work)}건")

# Phase 2: 웨이브별 (병렬 다운로드+파싱) → (분할 임베딩+upsert)
def embed_upsert(chunks):
    # ChromaDB 최대 배치(5461) 회피 + 임베딩(GPU)·upsert(CPU) 오버랩.
    # writer 스레드 1개만 upsert(동시 upsert 없음) → GPU가 다음 묶음 임베딩하는 동안 저장.
    import threading, queue
    q = queue.Queue(maxsize=2)   # 백프레셔: 임베딩 버퍼 최대 2묶음
    err = []
    def _writer():
        while True:
            item = q.get()
            if item is None:
                q.task_done(); break
            ids, texts, embs, metas = item
            try:
                add_batch(ids, texts, embs, metas)
            except Exception as e:
                err.append(e)
            finally:
                q.task_done()
    t = threading.Thread(target=_writer, daemon=True); t.start()
    n = 0
    for s in range(0, len(chunks), UPSERT_BATCH):
        if err:
            break
        sub = chunks[s:s + UPSERT_BATCH]
        texts = [c["text"] for c in sub]
        embs = embed_texts(texts, show_progress=False)
        q.put(([c["id"] for c in sub], texts, [e.tolist() for e in embs],
               [c["metadata"] for c in sub]))
        n += len(sub)
    q.put(None); t.join()
    if err:
        raise err[0]
    return n

done = emb = 0
for w in range(0, len(work), WAVE_SIZE):
    wave = work[w:w + WAVE_SIZE]
    collected = []
    with ThreadPoolExecutor(N_WORKERS) as ex:
        futs = {ex.submit(collect_report, rec, rep): (rec, rep) for rec, rep in wave}
        for fut in as_completed(futs):
            rec, rep = futs[fut]
            try:
                collected.append(fut.result())
            except Exception as e:
                fails.append((rec["stock_code"], rep["period"], str(e)[:140]))
    all_chunks = [c for r in collected for c in r["chunks"]]
    if all_chunks:
        ids = [c["id"] for c in all_chunks]
        existing = set()
        for s in range(0, len(ids), 5000):
            try: existing |= get_existing_ids(ids[s:s+5000])
            except Exception: pass
        todo = [c for c in all_chunks if c["id"] not in existing]
        if todo:
            emb += embed_upsert(todo)
    # 임베딩 성공 후 jsonl 저장 + progress (jsonl 존재 ⟺ 임베딩 완료 보장)
    for r in collected:
        if r["records"]:
            with open(jsonl_path(r["market"], r["stock"], r["corp_name"], r["year"], r["plabel"]),
                      "w", encoding="utf-8") as f:
                for rr in r["records"]:
                    f.write(json.dumps(rr, ensure_ascii=False) + "\n")
        progress[r["key"]] = "done"; done += 1
    PROGRESS_LOG.write_text(json.dumps(progress, ensure_ascii=False), encoding="utf-8")
    print(f"  웨이브 {w//WAVE_SIZE+1}: 보고서 {done}/{len(work)} | 임베딩 누적 {emb} 청크 | {time.time()-t0:.0f}s")

print(f"\\n=== 완료: 보고서 {done}건 · 임베딩 {emb} 청크 · 실패 {len(fails)}건 ({time.time()-t0:.0f}s) ===")


In [ ]:
# ── 6) 검증 ──────────────────────────────────────────────────────────
import glob
print("ChromaDB:", get_stats())
print("실패 샘플:", fails[:10])
q1 = glob.glob(str(KOSPI_DIR / "*_2026_q1_chunks.jsonl")) + glob.glob(str(KOSDAQ_DIR / "*_2026_q1_chunks.jsonl"))
print(f"2026 Q1 jsonl: {len(q1)}건")
if q1:
    rec = json.loads(open(q1[0], encoding="utf-8").readline())
    print("샘플:", {k: rec[k] for k in ("id","report_nm","report_kind","report_type","rcept_no")})
